# Gym Churn: Churn Prediction

This notebook trains two binary classification models, logistic regression and random forest, to predict which Model Fitness customers will churn in the following month, and compares them on accuracy, precision and recall.

In [1]:
# Libraries for data handling, modeling and evaluation
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)  # show all columns in outputs

## 1. Data Preparation for Modeling

In [2]:
# Load the cleaned dataset prepared in notebook 01
gym_data = pd.read_csv('../data/gym_churn_clean.csv')
gym_data.head()

,gender,near_location,partner,promo_friends,phone,contract_period,group_visits,age,avg_additional_charges_total,month_to_end_contract,lifetime,avg_class_frequency_total,avg_class_frequency_current_month,churn
0,1,1,1,1,0,6,1,29,14.227470,5,3,0.020398,0.000000,0
1,0,1,0,0,1,12,1,31,113.202938,12,7,1.922936,1.910244,0
2,0,1,1,0,1,1,0,28,129.448479,1,2,1.859098,1.736502,0
3,0,1,1,1,1,12,1,33,62.669863,12,2,3.205633,3.357215,0
4,1,1,1,1,1,1,0,26,198.362265,1,3,1.113884,1.120078,0


In [3]:
# Features (X) and target (y)
# One feature from each highly correlated pair is dropped to avoid multicollinearity
X = gym_data.drop(
    columns=['churn', 'month_to_end_contract', 'avg_class_frequency_total'])
y = gym_data['churn']

print(X.shape, y.shape)

(4000, 11) (4000,)


In [ ]:
# Split into training and validation sets (80/20)
X_train, X_valid, y_train, y_valid = train_test_split(
    # to ensure the same proportion of churned and non-churned customers in both sets
    X, y, test_size=0.2, random_state=0, stratify=y)

print(X_train.shape, X_valid.shape)

(3200, 11) (800, 11)


In [5]:
# Standardize features: fit the scaler on the training set only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

### Observations
- The target variable is `churn`; all other features are used as predictors, except `month_to_end_contract` and `avg_class_frequency_total`, which were dropped to avoid multicollinearity (0.97 and 0.95 correlation with `contract_period` and `avg_class_frequency_current_month`).
- The data was split into training (80%) and validation (20%) sets, stratified by churn to keep the same churn rate in both.
- Features were standardized with a scaler fitted on the training set only, so the validation set remains unseen by the model.

## 2. Model Training

Two binary classification models are trained to predict churn: logistic regression, a simple and interpretable linear model, and random forest, an ensemble of decision trees that can capture non-linear relationships. Both are evaluated on the validation set using accuracy, precision and recall.

### 2.1 Logistic Regression